In [ ]:
%cd ../..
import os
import polars as pl
import numpy as np
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, BertTokenizerFast

from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

random.seed(4)
np.random.seed(4)
torch.manual_seed(4)
torch.cuda.manual_seed_all(4)

In [ ]:
data_path = "/scratch/scratch0/BIMCV-R"

metadata = pl.read_csv(os.path.join(data_path, "curated_ct_report_path_En.csv"))
metadata = metadata.with_columns((pl.col("path").str.replace(".nii.gz", "")).alias("Path"))
metadata = metadata.select(["PatientID", "Path", "Report", "Report_en", "Labels"])

In [ ]:
embeddings_path = "/scratch/scratch0/embeddings/BIMCV-R"
all_ids_embed = os.listdir(embeddings_path)
all_ids_embed = [f.replace(".pth", "") for f in all_ids_embed]

In [ ]:
train_size=0.8

all_ids_metadata = metadata["PatientID"].unique().to_list()
ids_to_path = {}
for row in metadata.iter_rows():
    sid = row[0]
    path = row[1]
    if sid in ids_to_path.keys():
        ids_to_path[sid].append(path)
    else:
        ids_to_path[sid] = [path]


all_ids_with_embeddings = [sid for sid in all_ids_metadata if any(os.path.exists(os.path.join(embeddings_path, f"{path}.pth")) for path in ids_to_path[sid])]
random.shuffle(all_ids_with_embeddings)

num_ids = len(all_ids_with_embeddings)
num_train = int(train_size * num_ids)
train_patids = all_ids_with_embeddings[:num_train]

train_ids, val_ids = [], []
for row in metadata.iter_rows():
    sid = row[0]
    path = row[1]
    if sid in train_patids:
        train_ids.append(path)
    else:
        val_ids.append(path)

In [ ]:
def get_embedding(mapid):
    x = torch.load(os.path.join(embeddings_path, f"{mapid}.pth"), mmap=True)
    return x["cls"]

class BIMCV_R(Dataset):
    def __init__(self, ids, labels_df):
        self.labels_df = labels_df.filter(pl.col("Path").is_in(ids))

    def __len__(self):
        return len(self.labels_df)
    
    def __getitem__(self, idx):
        row = self.labels_df.row(idx)
        mapid = row[1]
        report = row[3]

        return get_embedding(mapid), report
    
def collate_fn_text(batch):
    img_embed = [x[0] for x in batch]  # each is (N, D)
    reports = [x[1] for x in batch]
    sizes = [x.shape[0] for x in img_embed]
    max_len = max(sizes)
    B = len(img_embed)
    D = img_embed[0].shape[1]
    mask = torch.zeros(B, max_len, dtype=torch.bool)
    embed_stacked = torch.zeros(B, max_len, D, dtype=img_embed[0].dtype)

    for i, s in enumerate(sizes):
        mask[i, :s] = True
        embed_stacked[i, :s] = img_embed[i]

    return embed_stacked, mask, reports

In [ ]:
embed_dim = 768
temperature = 0.07
device = torch.device("cuda")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

text_encoder = BertModel.from_pretrained("StanfordAIMI/RadBERT")
tokenizer = BertTokenizerFast.from_pretrained("StanfordAIMI/RadBERT")

In [ ]:
def contrastive_loss(text_emb, img_emb, temp):
    text_emb = F.normalize(text_emb, dim=1)
    img_emb = F.normalize(img_emb, dim=1)

    logits = torch.matmul(text_emb, img_emb.t())  # (B, B)
    logits = logits / temp

    labels = torch.arange(logits.size(0), device=logits.device)

    loss_t2i = F.cross_entropy(logits, labels)
    loss_i2t = F.cross_entropy(logits.t(), labels)

    return (loss_t2i + loss_i2t) / 2

In [ ]:
train_dataset = BIMCV_R(train_ids, metadata)
val_dataset = BIMCV_R(val_ids, metadata)

batch_size = 64
train_dataloader = DataLoader(train_dataset, num_workers=2, pin_memory=True, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_text)
val_dataloader = DataLoader(val_dataset, num_workers=2, pin_memory=True, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_text)

print(len(train_dataloader), len(val_dataloader))

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)
    
class AttentionPool(nn.Module):
    def __init__(self, embed_dim: int):
        super().__init__()
        self.query = nn.Parameter(torch.zeros(1, embed_dim))

        self.attention_net = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.Tanh(), nn.Linear(embed_dim, 1)
        )

        nn.init.xavier_uniform_(self.query)

    def forward(self, x, mask=None):
        B, N, D = x.shape

        query_expanded = self.query.expand(B, N, -1)

        combined = x + query_expanded

        raw_scores = self.attention_net(combined)

        if mask is not None:
            raw_scores = raw_scores.masked_fill(~mask.unsqueeze(-1), float("-inf"))

        attention_weights = torch.softmax(raw_scores, dim=1)
        weighted_sum = torch.sum(x * attention_weights, dim=1)

        return weighted_sum

text_proj = ProjectionHead(embed_dim, embed_dim)
vision_proj = AttentionPool(embed_dim=embed_dim) # (B, N, D) -> (B, D)

In [ ]:
for param in text_encoder.embeddings.parameters():
    param.requires_grad = False

for layer in text_encoder.encoder.layer[:6]:
    for param in layer.parameters():
        param.requires_grad = False

# also try layerwise lr decay

In [ ]:
text_encoder.to(device) # type: ignore
text_proj.to(device)
vision_proj.to(device)

bert_params = list(text_encoder.named_parameters())
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {"params":[p for n,p in text_encoder.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay":1e-2, "lr":2e-5},
    {"params":[p for n,p in text_encoder.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay":0.0, "lr":2e-5},
    {"params": text_proj.parameters(), "lr":1e-4, "weight_decay":1e-2},
    {"params": vision_proj.parameters(), "lr":1e-4, "weight_decay":1e-2},
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

In [ ]:
def evaluate_text_to_image(model_components, dataloader, device, ks=[1,5,10]):
    text_encoder, text_proj, vision_proj = model_components
    text_encoder.eval()
    text_proj.eval()
    vision_proj.eval()

    all_text_embs, all_img_embs = [], []
    running_loss = 0.0
    with torch.no_grad():
        for img_embs, masks, reports in tqdm(dataloader, desc="eval"):
            img_embs = img_embs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            encoding = tokenizer(
                list(reports),
                padding=True, truncation=True, max_length=128,
                return_tensors="pt"
            ).to(device)
            last = text_encoder(**encoding).last_hidden_state
            text_out = last[:,0,:]
            tproj = text_proj(text_out)

            vproj = vision_proj(img_embs, masks)

            loss = contrastive_loss(tproj, vproj, temperature)

            running_loss += loss.item() * img_embs.size(0)

            all_text_embs.append(tproj.cpu())
            all_img_embs.append(vproj.cpu())

    mean_loss = running_loss / len(dataloader.dataset)

    all_text_embs = torch.cat(all_text_embs, dim=0)
    all_img_embs = torch.cat(all_img_embs, dim=0)

    all_text_embs = torch.nn.functional.normalize(all_text_embs, dim=1)
    all_img_embs = torch.nn.functional.normalize(all_img_embs, dim=1)

    sims = all_text_embs @ all_img_embs.T
    sims = sims.cpu().numpy()

    ranks = []
    for i in range(sims.shape[0]):
        ranking = np.argsort(-sims[i])
        rank = np.where(ranking == i)[0][0]
        ranks.append(rank)

    ranks = np.array(ranks)
    results = {}
    for k in ks:
        results[f"R@{k}"] = np.mean(ranks < k) * 100
    results["MeanRank"] = ranks.mean() + 1
    results["MedianRank"] = np.median(ranks) + 1
    results["loss"] = mean_loss

    return results

In [ ]:
def train_one_epoch(model_components, dataloader, optimizer, device):
    text_encoder, text_proj, vision_proj = model_components
    text_encoder.train()
    text_proj.train()
    vision_proj.train()

    running_loss = 0.0
    for img_embs, masks, reports in tqdm(dataloader, desc="train"):
        img_embs = img_embs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        encoding = tokenizer(
            list(reports),
            padding=True, truncation=True, max_length=128,
            return_tensors="pt"
        ).to(device)

        last = text_encoder(**encoding).last_hidden_state
        text_out = last[:,0,:]
        tproj = text_proj(text_out)
        vproj = vision_proj(img_embs, masks)

        loss = contrastive_loss(tproj, vproj, temperature)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * img_embs.size(0)

    return running_loss / len(dataloader.dataset)

In [ ]:
num_epochs = 50

for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        (text_encoder, text_proj, vision_proj), train_dataloader, optimizer, device
    )
    results = evaluate_text_to_image(
        (text_encoder, text_proj, vision_proj), val_dataloader, device
    )

    val_loss = results["loss"]

    metrics_str = " | ".join([f"{k}: {v:.2f}" for k, v in results.items() if k != "loss"])

    print(
        f"Epoch {epoch+1}/{num_epochs} "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| {metrics_str}"
    )